In [1]:
%matplotlib inline

import os
import sys
import math
import time

from sys import platform

sys.path.append('../../../')

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from importlib.metadata import version 

import tiktoken

import torch
import torch.nn as nn



%load_ext autoreload
%autoreload 2

from llm_from_scratch.CH4.gpt import GPTModel
from llm_from_scratch.CH4.llama import Llama2Model, Llama3Model
# from llm_from_scratch.CH4.blocks import RMSNorm, SiLU, MultiheadAttentionLlama, FeedForwardLlama, GroupedQueryAttention
from llm_from_scratch.CH4.utils import calc_model_memory_size # precompute_rope_params, compute_rope, 
from llm_from_scratch.CH2.text_data_set import create_dataloader_v1
from llm_from_scratch.CH5.loss import calc_loss_batch
from llm_from_scratch.CH5.utils import find_highest_gradient, load_weights_into_llama3
from llm_from_scratch.CH5.optim import evaluate_model, generate_and_print_sample
from llm_from_scratch.CH5.utils import token_ids_to_text, text_to_token_ids, generate, Tokenizer
from llm_from_scratch.CH5.chat_format import ChatFormat, clean_text
print(f"torch version: {version('torch')}")

from llm_from_scratch.CH5.qwen35.block import GroupedQueryAttention, apply_rope, compute_rope_params, RMSNorm, FeedForward
from llm_from_scratch.CH5.qwen35.module import Qwen3_5GateDeltaNet
from llm_from_scratch.CH5.qwen35.qwen import KVCache, Qwen3_5LinearAttentionCache, Qwen3_5Model, _Qwen3_5ConfigAdapter
from llm_from_scratch.CH5.qwen35.utils import load_weights_into_qwen3_5, generate_text_basic_stream

torch version: 2.12.0


In [2]:
from importlib.metadata import version
pkgs=[
    "huggingface_hub", # to download pretrained weights
    "tokenizers",      # to implement the tokenizer
    "torch",           # to implement the model
]
for p in pkgs: print(f"{p} version: {version(p)}")

huggingface_hub version: 1.14.0
tokenizers version: 0.23.1
torch version: 2.12.0


In [3]:
USE_MODEL='Qwen3.5-0.8B'

In [4]:
# Qwen3.5-0.8B text configuration 
# d_out=n_heads*head_dim=2048
QWEN3_5_CONFIG={
    "vocab_size": 248_320,
    "context_length":262_144,
    "emb_dim":1_024,
    "n_heads":8,
    "n_layers":24,
    "hidden_dim": 3_584,
    "head_dim":256,
    "qk_norm":True,
    "n_kv_groups":2,
    "rope_base":10_000_000.0,
    "partial_rotary_factor":0.25,
    "rms_norm_eps":1e-6,
    "linear_conv_kernel_dim":4,
    "linear_key_head_dim":128,
    "linear_value_head_dim":128,
    "linear_num_key_heads":16,
    "linear_num_value_heads":16,
    "dtype":torch.bfloat16,
    "layer_types": [
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
    ],
}

torch.manual_seed(123)
model=Qwen3_5Model(QWEN3_5_CONFIG)
model

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Qwen3_5Model(
  (tok_emb): Embedding(248320, 1024)
  (trf_blocks): ModuleList(
    (0-2): 3 x TransformerBlock(
      (token_mixer): Qwen3_5GateDeltaNet(
        (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144, bias=False)
        (norm): Qwen3_5RMSNormGated()
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (in_proj_qkv): Linear(in_features=1024, out_features=6144, bias=False)
        (in_proj_z): Linear(in_features=1024, out_features=2048, bias=False)
        (in_proj_b): Linear(in_features=1024, out_features=16, bias=False)
        (in_proj_a): Linear(in_features=1024, out_features=16, bias=False)
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3584, bias=False)
        (fc2): Linear(in_features=1024, out_features=3584, bias=False)
        (fc3): Linear(in_features=3584, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
    (3): 

In [5]:
# check forward pass before continuing
outs=model(torch.tensor([1,2,3]).unsqueeze(0))

In [6]:
total_params=sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")
# account for weight tying
total_params_normalized=total_params-model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

Total number of parameters: 1,006,672,704

Total number of unique parameters: 752,393,024


In [7]:
print(f"float32 (Pytorch default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16 (Pytorch default): {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

float32 (Pytorch default): 7.63 GB
bfloat16 (Pytorch default): 3.81 GB


In [8]:
device=torch.device('cuda') if torch.cuda.is_available() else torch.device('mps') if torch.mps.is_available() else torch.device('cpu')

In [9]:
import json
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, snapshot_download

repo_id="Qwen/Qwen3.5-0.8B"
local_dir='/Users/reaungamornrat.sureerat/data/Qwen/'+Path(repo_id).parts[-1]
if not os.path.isdir(local_dir): os.makedirs(local_dir)

repo_dir=snapshot_download(repo_id=repo_id, local_dir=local_dir)
print(f"{repo_dir=}")

/Users/reaungamornrat.sureerat/miniforge3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 13 files: 100%|██████████████████████████████████████████████████████████████| 13/13 [00:00<00:00, 1229.31it/s]

repo_dir='/Users/reaungamornrat.sureerat/data/Qwen/Qwen3.5-0.8B'


In [10]:
index_path=os.path.join(repo_dir, "model.safetensors.index.json")
with open(index_path, "r") as f: index=json.load(f)
print(f"{len(index)=}, {index.keys()}")

len(index)=2, dict_keys(['metadata', 'weight_map'])


In [11]:
type(index['weight_map']), index['weight_map'].keys()

(dict,
 dict_keys(['model.language_model.embed_tokens.weight', 'model.visual.merger.linear_fc1.weight', 'model.language_model.layers.14.linear_attn.in_proj_qkv.weight', 'model.language_model.layers.12.linear_attn.in_proj_qkv.weight', 'model.language_model.layers.18.linear_attn.in_proj_qkv.weight', 'model.language_model.layers.13.linear_attn.in_proj_qkv.weight', 'model.language_model.layers.2.linear_attn.in_proj_qkv.weight', 'model.language_model.layers.4.linear_attn.in_proj_qkv.weight', 'model.language_model.layers.20.linear_attn.in_proj_qkv.weight', 'model.language_model.layers.1.linear_attn.in_proj_qkv.weight', 'model.language_model.layers.10.linear_attn.in_proj_qkv.weight', 'model.language_model.layers.16.linear_attn.in_proj_qkv.weight', 'model.language_model.layers.17.linear_attn.in_proj_qkv.weight', 'model.language_model.layers.5.linear_attn.in_proj_qkv.weight', 'model.language_model.layers.6.linear_attn.in_proj_qkv.weight', 'model.language_model.layers.8.linear_attn.in_proj_qkv.w

In [12]:
weights_dict={}
for i, filename in enumerate(sorted(set(index['weight_map'].values()))):
    if i<5: print(i, filename)
    shard_path=os.path.join(repo_dir, filename)
    shard=load_file(shard_path)
    weights_dict.update(shard) # add {key:value} to weights_dict

load_weights_into_qwen3_5(model,QWEN3_5_CONFIG, weights_dict)
model.to(device);
del weights_dict

0 model.safetensors-00001-of-00001.safetensors
Model uses weight tying


In [13]:
from llm_from_scratch.CH5.qwen35.tokenizer import Qwen3_5Tokenizer

local_dir='/Users/reaungamornrat.sureerat/data/Qwen/'+Path(repo_id).parts[-1]
tokenizer_file_path=local_dir+"/tokenizer.json"

hf_hub_download(repo_id=repo_id, filename='tokenizer.json', local_dir=local_dir)

tokenizer=Qwen3_5Tokenizer(tokenizer_file_path=tokenizer_file_path, repo_id=repo_id, apply_chat_template=True, add_generation_prompt=True,
                          add_thinking=True)

In [14]:
prompt="Give me a short introduction to large language model"
input_token_ids=tokenizer.encode(prompt)
print(f"{input_token_ids=}")
text=tokenizer.decode(input_token_ids)
print(f"{text=}\n")
print(text)

input_token_ids=[248045, 846, 198, 33963, 728, 264, 2716, 16308, 310, 3349, 3992, 1558, 248046, 198, 248045, 74455, 198, 248068, 198]
text='<|im_start|>user\nGive me a short introduction to large language model<|im_end|>\n<|im_start|>assistant\n<think>\n'

<|im_start|>user
Give me a short introduction to large language model<|im_end|>
<|im_start|>assistant
<think>



In [19]:
prompt="Give me a short introduction to large language model"

input_token_ids=tokenizer.encode(prompt)
input_token_ids_tensor=torch.tensor(input_token_ids, device=device).unsqueeze(0)
print(f"{len(input_token_ids)=}, {input_token_ids_tensor.shape=}")

if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

start_time=time.perf_counter()
generated_tokens=0

for i, token in enumerate(generate_text_basic_stream(model=model,token_ids=input_token_ids_tensor, max_new_tokens=500, 
                                        eos_token_id=tokenizer.eos_token_id)):
    # print(f'token {i=}',"-"*20)
    generated_tokens+=1
    token_id=token.squeeze(0).tolist()
    # print(f"\t{len(token_id)=}")
    # print(f"{tokenizer.decode(token_id)=}", flush=True) #end=""
    print(tokenizer.decode(token_id), flush=True, end="")

elapsed=time.perf_counter()-start_time
tokens_per_sec=generated_tokens/elapsed if elapsed > 0 else 0.
print(f"\n\nGeneraion speed: {tokens_per_sec:.2f} tokens/sec")

if torch.cuda.is_available():
    def calc_gpu_gb(x): return f"{x/1024/1024/1024:.2f} GB"
    print(f"GPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")

len(input_token_ids)=19, input_token_ids_tensor.shape=torch.Size([1, 19])
Thinking Process:

1.  **Analyze the Request:**
    *   **Topic:** Large Language Model (LLM).
    *   **Task:** Give a short introduction.
    *   **Constraint:** "Short" (implies concise, clear, not a full essay).

2.  **Identify Key Concepts:**
    *   What is it? A type of AI.
    *   What does it do? Understands, generates text, reasoning, etc.
    *   How does it work? (Neural networks, transformers, etc.)
    *   Why is it important? (Chatbots, content creation, etc.)
    *   *Self-Correction/Refinement:* Keep it simple and accessible. Avoid overly technical jargon unless necessary.

3.  **Drafting the Content:**
    *   *Definition:* An AI model that processes text and generates text.
    *   *Mechanism:* Uses neural networks (specifically transformers).
    *   *Function:* Understands context, language, and reasoning.
    *   *Examples:* Chatbots, writing, coding.
    *   *Future:* More advanced, better 

In [22]:
import time

prompt="A shop gives a 20% discount, then adds 10% tax. Is the final price higher or lower than the original? By how much?"

input_token_ids=tokenizer.encode(prompt) # list of token indices
input_token_ids_tensor=torch.tensor(input_token_ids, device=device).unsqueeze(0)

if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
start_time=time.perf_counter()
generated_tokens=0

for token in generate_text_basic_stream(model=model, token_ids=input_token_ids_tensor, max_new_tokens=500, 
                                        eos_token_id=tokenizer.eos_token_id):
    generated_tokens+=1
    token_id=token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end="", flush=True)

elapsed=time.perf_counter()-start_time
tokens_per_sec=generated_tokens/elapsed if elapsed>0 else 0.
print(f"\n\nGeneration speed: {tokens_per_sec:.2f} tokens/sec")

if torch.cuda.is_available():
    def calc_gpu_gb(x): return f"{x/1024/1024/1024:.2f} GB"
    print(f"GPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")

Here's a thinking process that leads to the solution:

1.  **Analyze the Request:**
    *   **Scenario:** A shop applies two discounts and a tax.
    *   **Discount:** 20% off the original price.
    *   **Tax:** 10% added on top of the discounted price.
    *   **Question:** Is the final price higher or lower than the original? By how much?

2.  **Define Variables:**
    *   Let $P$ be the original price.

3.  **Step-by-Step Calculation:**

    *   *Step 1: Apply the 20% discount.*
        *   Discount amount = $0.20 \times P$
        *   Final price after discount = $P - 0.20P$
        *   Final price after discount = $0.80P$

    *   *Step 2: Apply the 10% tax.*
        *   Tax amount = $0.10 \times (\text{Final price after discount})$
        *   Tax amount = $0.10 \times (0.80P)$
        *   Tax amount = $0.08P$
        *   Final price after tax = Final price after discount + Tax amount
        *   Final price after tax = $0.80P + 0.08P$
        *   Final price after tax = $0.88P$

In [20]:
type(input_token_ids)

list